In [28]:
from groq import Groq
import json
from dotenv import load_dotenv
from datetime import datetime, timedelta
import os.path

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from toolhouse import Toolhouse
from typing import List, Optional
import logging


load_dotenv()

import logging
import traceback
from functools import wraps

# Set up logging with detailed format
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s - [%(lineno)d]',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

def trace(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        func_name = func.__name__
        print(f"\n>>> ENTER {func_name}")
        print(f">>> Args: {args}")
        print(f">>> Kwargs: {kwargs}")
        try:
            result = func(*args, **kwargs)
            print(f"<<< EXIT {func_name}")
            if isinstance(result, str) and len(result) > 200:
                print(f"<<< Result: {result[:200]}...")
            else:
                print(f"<<< Result: {result}")
            return result
        except Exception as e:
            print(f"!!! ERROR in {func_name}: {str(e)}")
            print(traceback.format_exc())
            raise
    return wrapper


In [29]:
# If modifying these scopes, delete the file token.json.
SCOPES = ["https://www.googleapis.com/auth/calendar"]


creds = None
  # The file token.json stores the user's access and refresh tokens, and is
  # created automatically when the authorization flow completes for the first
  # time.
if os.path.exists("token.json"):
    creds = Credentials.from_authorized_user_file("token.json", SCOPES)
# If there are no (valid) credentials available, let the user log in.
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow = InstalledAppFlow.from_client_secrets_file(
            "credentials.json", SCOPES
        )
        creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())



In [30]:
# Initialize the Groq client 
client = Groq()
th = Toolhouse()
# Define models
ROUTING_MODEL = "llama3-70b-8192"
#TOOL_USE_MODEL = "llama-3.3-70b-versatile"
TOOL_USE_MODEL = "llama3-groq-70b-8192-tool-use-preview"
GENERAL_MODEL = "llama3-70b-8192"

2024-12-28 13:13:00,719 - httpx - DEBUG - load_ssl_context verify=True cert=None trust_env=True http2=False - [82]
2024-12-28 13:13:00,732 - httpx - DEBUG - load_verify_locations cafile='/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/certifi/cacert.pem' - [148]


In [31]:
relevant_fields = ["start", "end", "summary", "description"]

@th.register_local_tool("get_calendar_events")
@trace
def get_calendar_events(start_time: str, end_time: str):
    """Get calendar events between start and end time"""
    print("\n=== get_calendar_events Parameters ===")
    print(f"start_time: {start_time}")
    print(f"end_time: {end_time}")
    print("=====================================\n")
    
    try:
        service = build("calendar", "v3", credentials=creds)
        calendar_list = service.calendarList().list().execute()
        events = []
        for calendar in calendar_list.get("items", []):
            calendar_id = calendar["id"]
            print(f"Fetching events for calendar: {calendar['summary']}")
            
            events_result = service.events().list(
                calendarId=calendar_id,
                timeMin=start_time,
                timeMax=end_time,
                singleEvents=True,
                orderBy="startTime",
            ).execute()
            filtered_events = [{key: item[key] for key in relevant_fields if key in item} 
                             for item in events_result.get("items", [])]
            events.append({
                "calendar_name": calendar["summary"],
                "events": filtered_events
            })
        return json.dumps({"result": events})
    except Exception as e:
        print(f"Error in get_calendar_events: {str(e)}")
        return json.dumps({"error": str(e)})

@th.register_local_tool("create_calendar_event")
@trace
def create_calendar_event(
    summary: str,
    description: str,
    start_time: str, 
    end_time: str, 
    recurrence: object, 
    location:str=None, 
    attendees:Optional[List[object]]=[], 
    reminders:Optional[List[object]]=[]
    ):
    """Create a calendar event with the specified parameters"""
    print("\n=== create_calendar_event Parameters ===")
    print(f"summary: {summary}")
    print(f"description: {description}")
    print(f"start_time: {start_time}")
    print(f"end_time: {end_time}")
    print(f"recurrence: {recurrence}")
    print(f"location: {location}")
    print(f"attendees: {attendees}")
    print(f"reminders: {reminders}")
    print("======================================\n")
    
    try:
        # Verify and refresh OAuth credentials if needed
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                print("Refreshing expired credentials...")
                creds.refresh(Request())
            else:
                print("No valid credentials")
                flow = InstalledAppFlow.from_client_secrets_file(
                    "credentials.json", SCOPES)
                creds = flow.run_local_server(port=0)
                with open("token.json", "w") as token:
                    token.write(creds.to_json())
        
        print("Building calendar service with OAuth...")
        service = build("calendar", "v3", credentials=creds)
        
        # Format the time properly
        start_time_obj = {
            'dateTime': start_time,
            'timeZone': 'America/New_York'
        }
        end_time_obj = {
            'dateTime': end_time,
            'timeZone': 'America/New_York'
        }
        
        event = {
            'summary': summary,
            'location': location,
            'description': description,
            'start': start_time_obj,
            'end': end_time_obj,
            'recurrence': recurrence,
            'attendees': attendees,
            'reminders': reminders,
        }
        print(f"Event object constructed: {json.dumps(event, indent=2)}")
        
        print("Inserting event into calendar...")
        event = service.events().insert(
            calendarId='primary',
            body=event
        ).execute()
        
        print(f"Event created successfully with ID: {event.get('id')}")
        return json.dumps("event insertion successful")
        
    except HttpError as e:
        error_details = {
            "error": "Google Calendar API Error",
            "status_code": e.resp.status,
            "reason": e.resp.reason,
            "error_details": json.loads(e.content.decode('utf-8'))
        }
        print(f"HTTP Error: {json.dumps(error_details, indent=2)}")
        return json.dumps({"error": error_details})
    except Exception as e:
        print(f"Error creating calendar event: {str(e)}")
        print(f"Full traceback:\n{traceback.format_exc()}")
        return json.dumps({"error": str(e)})

In [32]:
@th.register_local_tool("create_calendar_event")
@trace
def create_calendar_event(
    summary: str,
    description: str,
    start_time: str, 
    end_time: str, 
    recurrence: object=None, 
    location:str=None, 
    attendees:Optional[List[object]]=[], 
    reminders:Optional[List[object]]=[]
    ):
    """Create a calendar event with the specified parameters"""
    print("\n=== create_calendar_event Parameters ===")
    print(f"summary: {summary}")
    print(f"description: {description}")
    print(f"start_time: {start_time}")
    print(f"end_time: {end_time}")
    print(f"recurrence: {recurrence}")
    print(f"location: {location}")
    print(f"attendees: {attendees}")
    print(f"reminders: {reminders}")
    print("======================================\n")

    try:
        print("Building calendar service with OAuth...")
        service = build("calendar", "v3", credentials=creds)
        
        # Format the time properly
        start_time_obj = {
            'dateTime': start_time,
            'timeZone': 'America/New_York'
        }
        end_time_obj = {
            'dateTime': end_time,
            'timeZone': 'America/New_York'
        }
        
        event = {
            'summary': summary,
            'location': location,
            'description': description,
            'start': start_time_obj,
            'end': end_time_obj,
            'recurrence': recurrence,
            'attendees': attendees,
            'reminders': reminders,
        }
        print(f"Event object constructed: {json.dumps(event, indent=2)}")
        
        print("Inserting event into calendar...")
        event = service.events().insert(
            calendarId='primary',
            body=event
        ).execute()
        
        print(f"Event created successfully with ID: {event.get('id')}")
        return json.dumps("event insertion successful")
        
    except HttpError as e:
        error_details = {
            "error": "Google Calendar API Error",
            "status_code": e.resp.status,
            "reason": e.resp.reason,
            "error_details": json.loads(e.content.decode('utf-8'))
        }
        print(f"HTTP Error: {json.dumps(error_details, indent=2)}")
        return json.dumps({"error": error_details})
    except Exception as e:
        print(f"Error creating calendar event: {str(e)}")
        print(f"Full traceback:\n{traceback.format_exc()}")
        return json.dumps({"error": str(e)})


In [38]:

local_tools = [
        {
            "type": "function",
            "function": {
                "name": "get_calendar_events",
                "description": "get events of the user's calendar, at the time of the request, it is currently," + datetime.utcnow().isoformat() + "Z. Determine the start and end window given their query",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "start_time": {
                            "type": "string",
                            "description": "The start of the window where the retrieved events will fall between",
                        },
                        "end_time": {
                            "type": "string",
                            "description": "The end of the window where the retrieved events will fall between"
                        }
                    },
                    "required": ["start_time", "end_time"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "create_calendar_event",
                "description": "make an event and put it in the user's calendar,at the time of the request, it is currently," + datetime.utcnow().isoformat() + "Z. Determine the start and end window given their query",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "summary": {
                            "type": "string",
                            "description": "The title of the event as it appears in the calendar"
                        },
                        "description": {
                            "type": "string",
                            "description": "The detailed description of the event as it appears in the calendar"
                        },
                        "start_time": {
                            "type": "string",
                            "description": "The start of the window where the retrieved events will fall between",
                        },
                        "end_time": {
                            "type": "string",
                            "description": "The end of the window where the retrieved events will fall between"
                        },
                        "recurrence": {
                            "type": "list",
                            "description": 
                            """
                            Recurring events
Some events occur multiple times on a regular schedule, such as weekly meetings, birthdays, and holidays. Other than having different start and end times, these repeated events are often identical.

Events are called recurring if they repeat according to a defined schedule. Single events are non-recurring and happen only once.

Recurrence rule
The schedule for a recurring event is defined in two parts:

Its start and end fields (which define the first occurrence, as if this were just a stand-alone single event), and

Its recurrence field (which defines how the event should be repeated over time).

The recurrence field contains an array of strings representing one or several RRULE, RDATE or EXDATE properties as defined in RFC 5545.

The RRULE property is the most important as it defines a regular rule for repeating the event. It is composed of several components. Some of them are:

FREQ — The frequency with which the event should be repeated (such as DAILY or WEEKLY). Required.

INTERVAL — Works together with FREQ to specify how often the event should be repeated. For example, FREQ=DAILY;INTERVAL=2 means once every two days.

COUNT — Number of times this event should be repeated.

You can use either COUNT or UNTIL to specify the end of the event recurrence. Don't use both in the same rule.
UNTIL — The date or date-time until which the event should be repeated (inclusive).

BYDAY — Days of the week on which the event should be repeated (SU, MO, TU, etc.). Other similar components include BYMONTH, BYYEARDAY, and BYHOUR.

The RDATE property specifies additional dates or date-times when the event occurrences should happen. For example, RDATE;VALUE=DATE:19970101,19970120. Use this to add extra occurrences not covered by the RRULE.

The EXDATE property is similar to RDATE, but specifies dates or date-times when the event should not happen. That is, those occurrences should be excluded. This must point to a valid instance generated by the recurrence rule.

EXDATE and RDATE can have a time zone, and must be dates (not date-times) for all-day events.

Each of the properties may occur within the recurrence field multiple times. The recurrence is defined as the union of all RRULE and RDATE rules, minus the ones excluded by all EXDATE rules.

Here are some examples of recurrent events:

An event that happens from 6am until 7am every Tuesday and Friday starting from September 15th, 2015 and stopping after the fifth occurrence on September 29th:


...
"start": {
 "dateTime": "2015-09-15T06:00:00+02:00",
 "timeZone": "Europe/Zurich"
},
"end": {
 "dateTime": "2015-09-15T07:00:00+02:00",
 "timeZone": "Europe/Zurich"
},
"recurrence": [
 "RRULE:FREQ=WEEKLY;COUNT=5;BYDAY=TU,FR"
],
…
An all-day event starting on June 1st, 2015 and repeating every 3 days throughout the month, excluding June 10th but including June 9th and 11th:


...
"start": {
 "date": "2015-06-01"
},
"end": {
 "date": "2015-06-02"
},
"recurrence": [
 "EXDATE;VALUE=DATE:20150610",
 "RDATE;VALUE=DATE:20150609,20150611",
 "RRULE:FREQ=DAILY;UNTIL=20150628;INTERVAL=3"
],
…
Instances & exceptions
A recurring event consists of several instances: its particular occurrences at different times. These instances act as events themselves.

Recurring event modifications can either affect the whole recurring event (and all of its instances), or only individual instances. Instances that differ from their parent recurring event are called exceptions.

For example, an exception may have a different summary, a different start time, or additional attendees invited only to that instance. You can also cancel an instance altogether without removing the recurring event (instance cancellations are reflected in the event status).

Examples of how to work with recurring events and instances via the Google Calendar API can be found here.
                            """
                        },
                        "location": {
                            "type": "string",
                            "description": "The place where this event is taking place at"
                        },
                        "attendees": {
                            "type": "list",
                            "description": "a list of {'email': 'example_handle@example.com'}"
                        },
                        "reminders": {
                            "type": "object",
                            "description": "a reminder(s) is a" +
                            """
                            "reminders": {
                                "useDefault": boolean,
                                "overrides": [
                                {
                                    "method": string,
                                    "minutes": integer
                                }
                                ]
                            }
                            """
                        },
                    },
                    "required": ["start_time", "end_time", "summary", "description"],
                },
            },
        },
        
    ]

/var/folders/1s/2z3_qy8136s0zkc3jk82g1jr0000gr/T/ipykernel_79124/3822424269.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "description": "get events of the user's calendar, at the time of the request, it is currently," + datetime.utcnow().isoformat() + "Z. Determine the start and end window given their query",
/var/folders/1s/2z3_qy8136s0zkc3jk82g1jr0000gr/T/ipykernel_79124/3822424269.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "description": "make an event and put it in the user's calendar,at the time of the request, it is currently," + datetime.utcnow().isoformat() + "Z. Determine the start and end window given their query",


In [45]:
import time
from datetime import datetime



def run_model_with_debug(query):
    print(f"\n=== Debug Info for Query: {query} ===")
    print(f"Start Time: {datetime.now()}")
    start_time = time.time()
    
    messages = [
        {
            "role": "system",
            "content": "You are an agentic compound ai system to help people with adhd regulate their schedule and achieve their goals!, If there are any requests that deal with time, you need to determine the parameters yourself" +
            "You are not allowed to ask for information that the user does not provide you execute the given command no matter what"
        },
        {
            "role": "user",
            "content": query,
        }
    ]
    
    print("\nSending Messages:")
    for msg in messages:
        print(f"- {msg['role']}: {msg['content'][:100]}...")
    
    all_tools = th.get_tools() + local_tools
    
    try:
        print("\nMaking API Call...")
        response = client.chat.completions.create(
            model=TOOL_USE_MODEL,
            messages=messages,
            tools=all_tools,
            tool_choice="auto",
            max_tokens=4096
        )
        
        print(f"\nResponse received in {time.time() - start_time:.2f} seconds")
        print(f"Usage: {response.usage.total_tokens} tokens")
        
        if hasattr(response, 'tool_calls') and response.tool_calls:
            print("\nTool Calls Made:")
            for tool_call in response.tool_calls:
                print(f"- {tool_call.function.name}: {tool_call.function.arguments}")
        
        tool_run = th.run_tools(response)
        messages.extend(tool_run)
        
        return response.choices[0].message.content
        
    except Exception as e:
        print(f"\nError occurred after {time.time() - start_time:.2f} seconds:")
        print(f"Type: {type(e).__name__}")
        print(f"Message: {str(e)}")
        raise
    finally:
        print(f"\nTotal execution time: {time.time() - start_time:.2f} seconds")
        print("=== Debug Session End ===\n")

queries = [
    "Can you schedule a volleyball open gym january second 4:30 to 6:30? I am just going there to have fun at monta vista highschool"
]

# Add trace decorator to your functions
run_model_with_debug = trace(run_model_with_debug)
create_calendar_event = trace(create_calendar_event)
get_calendar_events = trace(get_calendar_events)

for query in queries:
    run_model_with_debug(query)
    print("\n\n")

2024-12-28 13:23:27,970 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): api.toolhouse.ai:443 - [1051]



>>> ENTER run_model_with_debug
>>> Args: ('Can you schedule a volleyball open gym january second 4:30 to 6:30? I am just going there to have fun at monta vista highschool',)
>>> Kwargs: {}

=== Debug Info for Query: Can you schedule a volleyball open gym january second 4:30 to 6:30? I am just going there to have fun at monta vista highschool ===
Start Time: 2024-12-28 13:23:27.969324

Sending Messages:
- system: You are an agentic compound ai system to help people with adhd regulate their schedule and achieve t...
- user: Can you schedule a volleyball open gym january second 4:30 to 6:30? I am just going there to have fu...


2024-12-28 13:23:29,146 - urllib3.connectionpool - DEBUG - https://api.toolhouse.ai:443 "POST /v1/get_tools HTTP/11" 200 None - [546]
2024-12-28 13:23:29,150 - groq._base_client - DEBUG - Request options: {'method': 'post', 'url': '/openai/v1/chat/completions', 'files': None, 'json_data': {'messages': [{'role': 'system', 'content': 'You are an agentic compound ai system to help people with adhd regulate their schedule and achieve their goals!, If there are any requests that deal with time, you need to determine the parameters yourselfYou are not allowed to ask for information that the user does not provide you execute the given command no matter what'}, {'role': 'user', 'content': 'Can you schedule a volleyball open gym january second 4:30 to 6:30? I am just going there to have fun at monta vista highschool'}], 'model': 'llama3-groq-70b-8192-tool-use-preview', 'max_tokens': 4096, 'tool_choice': 'auto', 'tools': [{'type': 'function', 'function': {'name': 'exa_web_search', 'description':


Making API Call...


2024-12-28 13:23:29,898 - httpcore.http11 - DEBUG - receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Sat, 28 Dec 2024 21:23:29 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'8f949c9bdfd3228b-SJC'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Cache-Control', b'private, max-age=0, no-store, no-cache, must-revalidate'), (b'Vary', b'Origin, Accept-Encoding'), (b'Via', b'1.1 google'), (b'alt-svc', b'h3=":443"; ma=86400'), (b'x-groq-region', b'us-west-1'), (b'x-ratelimit-limit-requests', b'14400'), (b'x-ratelimit-limit-tokens', b'15000'), (b'x-ratelimit-remaining-requests', b'14399'), (b'x-ratelimit-remaining-tokens', b'14879'), (b'x-ratelimit-reset-requests', b'6s'), (b'x-ratelimit-reset-tokens', b'484ms'), (b'x-request-id', b'req_01jg7je7czeg1ry95pyn11kt2t'), (b'Server', b'cloudflare'), (b'Content-Encoding', b'gzip')]) - [45]
2024-12-28 13:23:29,898 - httpx - INFO - HTTP Request:


Response received in 1.93 seconds
Usage: 5872 tokens

Total execution time: 1.93 seconds
=== Debug Session End ===

<<< EXIT run_model_with_debug
<<< Result: To schedule your event, I need to know the location in a format that can be used in a calendar event. Could you provide the full address of Monta Vista High School?
 Note that the provided function is...



